# Map-Reduce Using Send() API with LangChain Agents
### Graph-Native Dynamic Fan-Out and Reducers Embedding LangChain Agents inside a StateGraph

This notebook demonstrates the official **`Send()` API** pattern for Map-Reduce:
1. **LangChain Agents**: We build specialized agents for planning, research (with Tavily search), and synthesis.
2. **LangGraph StateGraph**: We embed these LangChain agents as graph nodes.
3. **`Send()` Conditional Fan-Out**: The graph dynamically spawns $N$ parallel research workers based on the runtime plan.
4. **State Reducers (`operator.add`)**: LangGraph's Pregel engine collects all worker outputs concurrently and feeds them to the reducer node.

---

### Architectural Flow

```
                       ┌──────────────────────────────────────────────┐
                       │   Orchestrator Node (LangChain LLM Chain)    │
                       └──────────────────────┬───────────────────────┘
                                              │
                    conditional edge returns [Send("worker", task) for ...]
                                              │
                      ┌───────────────────────┼───────────────────────┐
                      ▼                       ▼                       ▼
               ┌─────────────┐         ┌─────────────┐         ┌─────────────┐
               │ LangChain   │         │ LangChain   │         │ LangChain   │
               │ Agent Worker│         │ Agent Worker│         │ Agent Worker│
               │ (Entity 1)  │         │ (Entity 2)  │         │ (Entity 3)  │
               └──────┬──────┘         └──────┬──────┘         └──────┬──────┘
                      │                       │                       │
                      └───────────────────────┼───────────────────────┘
                                              │ (State Reducer: operator.add)
                                              ▼
                       ┌──────────────────────────────────────────────┐
                       │     Reducer Node (LangChain Reducer Agent)   │
                       └──────────────────────────────────────────────┘
```


## 1. Setup & Environment Variables
Configure API keys for Google Gemini (`GOOGLE_API_KEY` / `GEMINI_API_KEY`) and Tavily Search (`TAVILY_API_KEY`).


In [ ]:
import os
from getpass import getpass
from dotenv import load_dotenv

# Robustly load .env from current or parent directory
if os.path.exists(".env"):
    load_dotenv(dotenv_path=".env")
elif os.path.exists("../.env"):
    load_dotenv(dotenv_path="../.env")
else:
    load_dotenv()

if "GEMINI_API_KEY" not in os.environ and "GOOGLE_API_KEY" in os.environ:
    os.environ["GEMINI_API_KEY"] = os.environ["GOOGLE_API_KEY"]

print("✓ Environment configured successfully!")
print(f"Google API Key present: {'GEMINI_API_KEY' in os.environ or 'GOOGLE_API_KEY' in os.environ}")
print(f"Tavily API Key present: {'TAVILY_API_KEY' in os.environ}")


## 2. Research Tools
Define the real-time Tavily search tool that our LangChain worker agent will invoke.


In [ ]:
from langchain.tools import tool
from langchain_community.utilities.tavily_search import TavilySearchAPIWrapper

tavily_wrapper = TavilySearchAPIWrapper()

@tool
def search_web(query: str) -> str:
    """Searches the web for up-to-date technical, architectural, and business information."""
    try:
        results = tavily_wrapper.results(query, max_results=3)
        if not results:
            return f"No results found for query: '{query}'."
        
        output = f"Search Results for '{query}':\n"
        for i, res in enumerate(results, 1):
            output += f"Source {i}: {res.get('title', 'Untitled')}\n"
            output += f"URL: {res.get('url', 'N/A')}\n"
            output += f"Content: {res.get('content', '')}\n\n"
        return output
    except Exception as e:
        return f"Web search encountered an error: {str(e)}"

print("✓ Search tool instantiated.")


## 3. Schemas & State Definition with `operator.add`
Here we define the Pydantic schemas for the plan and the LangGraph `OverallState`.
Notice `research_results: Annotated[List[dict], operator.add]`—this tells LangGraph to append parallel worker results rather than overwriting them.


In [ ]:
import operator
from typing import TypedDict, List, Optional, Annotated
from pydantic import BaseModel, Field

class EntityTask(BaseModel):
    id: int = Field(description="Unique numeric identifier for ordering in the reduce step")
    entity_name: str = Field(description="Name of the competitor, product, or technology to analyze")
    focus_areas: List[str] = Field(description="Key criteria to evaluate: architecture, adoption, features, limitations")
    search_queries: List[str] = Field(description="2-3 specific search queries to gather fresh evidence")

class BenchmarkingPlan(BaseModel):
    market_domain: str = Field(description="The sector or technology category being analyzed")
    evaluation_criteria: List[str] = Field(description="Standardized criteria to evaluate across all competitors")
    entities: List[EntityTask] = Field(description="List of entities to research in parallel")

class OverallState(TypedDict):
    topic: str
    plan: Optional[BenchmarkingPlan]
    # State Reducer: parallel worker outputs are aggregated via operator.add
    research_results: Annotated[List[dict], operator.add]
    final_report: str

class WorkerPayload(TypedDict):
    task: dict
    criteria: List[str]

print("✓ Schemas and State defined.")


## 4. LangChain Agents Instantiation
We instantiate two pure LangChain agents:
1. `worker_agent`: Specializes in deep entity investigation using Tavily search.
2. `reducer_agent`: Specializes in synthesis and Markdown comparison matrix compilation.


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0)

WORKER_SYSTEM_PROMPT = """You are an Expert Technical Research Analyst.
Your task is to conduct an in-depth, factual investigation of a single company, tool, or framework.

Instructions:
1. Use the search_web tool with the provided queries to gather authoritative information.
2. Analyze the entity against the provided evaluation criteria.
3. Structure your response in clean Markdown with:
   - **Executive Overview**: What is it and its primary design philosophy?
   - **Key Capabilities & Architecture**: Technical strengths.
   - **Limitations / Trade-offs**: What are its weaknesses or common complaints?
   - **Target Audience / Best Fit**: When should an engineering team choose it?
4. Keep the report factual, technical, and concise (200-300 words). Cite sources at the end."""

worker_agent = create_agent(
    model=llm,
    tools=[search_web],
    system_prompt=WORKER_SYSTEM_PROMPT
)

REDUCER_SYSTEM_PROMPT = """You are the Principal Synthesis & Market Strategy Officer.
Your job is to read individual research profiles of competing entities and produce a comprehensive, publication-grade Comparative Benchmark Report.

Hard Requirements:
1. Title: Clean H1 (# [Market Domain] Comparative Benchmark Report).
2. Executive Summary: 100-150 words synthesizing the state of the market and major architectural trends.
3. Side-by-Side Comparison Matrix: A detailed Markdown Table comparing all analyzed entities across standard dimensions (Architecture, Core Strengths, Limitations, Best Use Case).
4. Deep-Dive Entity Profiles: Clean H2 sections for each entity, preserving key technical facts from the workers.
5. Strategic Recommendation & Verdict: Clear decision framework for teams choosing between these options.

Ensure all special characters like '$' are escaped as '\\$' for clean rendering."""

reducer_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=REDUCER_SYSTEM_PROMPT
)

print("✓ Worker Agent and Reducer Agent instantiated.")


## 5. Graph Nodes & Dynamic Fan-Out (`Send` API)
Here we define the nodes of the graph.
* `fanout_to_workers`: A conditional edge function returning a list of `Send("research_worker", payload)` objects.
* `research_worker_node`: Wraps `worker_agent.invoke()`.
* `reducer_node`: Synchronizes when all workers finish and wraps `reducer_agent.invoke()`.


In [ ]:
import time
from langgraph.types import Send
from langchain_core.messages import SystemMessage, HumanMessage
from rich.console import Console

console = Console()

ORCHESTRATOR_PROMPT = """You are a Principal Market Intelligence Architect.
Given a technology domain or comparison topic, produce a structured benchmarking plan:
1. Identify 3 to 4 leading or emerging solutions in this domain.
2. Define 3-5 standardized evaluation criteria (e.g. Architecture, Core Strengths, Ecosystem, Pricing/Licensing).
3. For each entity, specify tailored focus areas and 2 high-signal search queries.

Output MUST strictly adhere to the BenchmarkingPlan schema."""

def orchestrator_node(state: OverallState) -> dict:
    """Planner node: decomposes the topic into a structured BenchmarkingPlan."""
    topic = state["topic"]
    console.print(f"\n[bold cyan]1. Orchestrator analyzing topic: '{topic}'...[/bold cyan]")
    
    planner = llm.with_structured_output(BenchmarkingPlan)
    plan = planner.invoke([
        SystemMessage(content=ORCHESTRATOR_PROMPT),
        HumanMessage(content=f"Analyze and benchmark the following domain: {topic}")
    ])
    
    console.print(f"✓ Planned {len(plan.entities)} entities: {[e.entity_name for e in plan.entities]}")
    return {"plan": plan}

def fanout_to_workers(state: OverallState) -> List[Send]:
    """Conditional Edge: Maps each planned entity to a parallel worker node using the Send API."""
    plan = state["plan"]
    assert plan is not None, "Orchestrator must produce a plan before fanout."
    
    console.print(f"\n[bold magenta]🚀 FANNING OUT {len(plan.entities)} PARALLEL WORKERS VIA Send()...[/bold magenta]")
    return [
        Send(
            "research_worker",
            {
                "task": entity.model_dump(),
                "criteria": plan.evaluation_criteria
            }
        )
        for entity in plan.entities
    ]

def research_worker_node(payload: WorkerPayload) -> dict:
    """Worker node: executes the LangChain agent for an assigned entity."""
    task_dict = payload["task"]
    criteria = payload["criteria"]
    name = task_dict["entity_name"]
    
    start = time.time()
    console.print(f"[bold cyan]>>> STARTING WORKER NODE for: {name}[/bold cyan]")
    
    query = (
        f"Entity to research: {name}\n"
        f"Evaluation Criteria: {', '.join(criteria)}\n"
        f"Specific Focus Areas: {', '.join(task_dict.get('focus_areas', []))}\n"
        f"Suggested Search Queries: {task_dict.get('search_queries', [])}\n"
    )
    
    response = worker_agent.invoke({"messages": [{"role": "user", "content": query}]})
    last_msg = response["messages"][-1]
    content = last_msg.content[0]["text"] if isinstance(last_msg.content, list) else last_msg.content
    
    elapsed = time.time() - start
    console.print(f"[bold green]<<< COMPLETED WORKER NODE for: {name} in {elapsed:.2f}s[/bold green]")
    
    # Return to the state reducer (operator.add appends to list)
    return {
        "research_results": [{
            "id": task_dict["id"],
            "entity_name": name,
            "content": content,
            "elapsed_seconds": elapsed
        }]
    }

def reducer_node(state: OverallState) -> dict:
    """Reducer node: runs after barrier sync, sorting results and compiling the final report."""
    plan = state["plan"]
    assert plan is not None
    results = sorted(state.get("research_results", []), key=lambda x: x["id"])
    
    console.print(f"\n[bold yellow]🔄 REDUCER NODE: Synthesizing {len(results)} completed research profiles...[/bold yellow]")
    
    combined_research = f"Market Domain: {plan.market_domain}\n"
    combined_research += f"Evaluation Criteria: {', '.join(plan.evaluation_criteria)}\n\n"
    
    for r in results:
        combined_research += f"### Entity: {r['entity_name']}\n"
        combined_research += f"{r['content']}\n\n"
        combined_research += "---\n"
        
    response = reducer_agent.invoke({
        "messages": [{"role": "user", "content": combined_research}]
    })
    
    last_msg = response["messages"][-1]
    final_report = last_msg.content[0]["text"] if isinstance(last_msg.content, list) else last_msg.content
    console.print("[bold green]✓ REDUCE NODE COMPLETE![/bold green]")
    
    return {"final_report": final_report}

print("✓ Graph nodes and Send() fan-out defined.")


## 6. Build & Compile StateGraph
We wire the nodes and conditional edges together:
* `START` -> `orchestrator`
* `orchestrator` -> conditional edge (`fanout_to_workers` returning `Send`) -> `research_worker`
* `research_worker` -> `reducer`
* `reducer` -> `END`


In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(OverallState)

# Add nodes
builder.add_node("orchestrator", orchestrator_node)
builder.add_node("research_worker", research_worker_node)
builder.add_node("reducer", reducer_node)

# Add edges
builder.add_edge(START, "orchestrator")
# Dynamic conditional edge using Send()
builder.add_conditional_edges("orchestrator", fanout_to_workers, ["research_worker"])
# All worker branches fan in to the reducer
builder.add_edge("research_worker", "reducer")
builder.add_edge("reducer", END)

# Compile graph
app = builder.compile()

print("✓ StateGraph compiled successfully!")
print(app.get_graph().draw_ascii())


## 7. End-to-End Execution Run
Execute the compiled graph with a market query and save the synthesized benchmark report to disk.


In [ ]:
import re
from pathlib import Path

def save_report_to_disk(title: str, content: str) -> str:
    os.makedirs("reports", exist_ok=True)
    safe_title = re.sub(r'[^a-zA-Z0-9\s\_\-]', '', title).strip().replace(" ", "_")
    filepath = f"reports/{safe_title}.md"
    Path(filepath).write_text(content, encoding="utf-8")
    return filepath

topic = "AI Agent Frameworks: LangGraph, AutoGen, and CrewAI"

console.print("\n" + "="*80, style="bold blue")
console.print(f"🎯 EXECUTING SEND() MAP-REDUCE GRAPH FOR: '{topic}'", style="bold blue")
console.print("="*80, style="bold blue")

start_time = time.time()
final_state = app.invoke({
    "topic": topic,
    "plan": None,
    "research_results": [],
    "final_report": ""
})
total_time = time.time() - start_time

results = final_state.get("research_results", [])
sum_individual = sum(r.get("elapsed_seconds", 0) for r in results)

console.print("\n" + "─"*80, style="bold green")
console.print(f"⏱️  Total Graph Execution Time : {total_time:.2f} seconds")
console.print(f"⏱️  Sum of Individual Worker Times: {sum_individual:.2f} seconds")
if total_time > 0 and sum_individual > 0:
    console.print(f"⚡ Speedup Factor                : {sum_individual / total_time:.2f}x faster!", style="bold yellow")
console.print("─"*80, style="bold green")

# Save deliverable
report_content = final_state["final_report"]
saved_path = save_report_to_disk("Send_API_Agent_Frameworks_Benchmark", report_content)
console.print(f"\n💾 [bold green]Saved Benchmark Report to: {saved_path}[/bold green]")

print("\n" + "="*80)
print("EXECUTIVE BENCHMARK DELIVERABLE (PREVIEW):\n")
print(report_content[:1200] + "\n\n[... Full Report Saved on Disk ...]")
